In [4]:
# ============================================================
# IMPORTAÇÕES
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    balanced_accuracy_score
)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def formatar_numero(valor, casas=2):
    if pd.isna(valor):
        return "NA"
    return f"{valor:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def formatar_inteiro(valor):
    return f"{int(valor):,}".replace(",", ".")


def fig_plotly_to_html(fig):
    return pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs=False,
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "responsive": True,
            "displaylogo": False,
            "modeBarButtonsToRemove": ["lasso2d", "select2d"]
        }
    )


def calcular_entropia_shannon(probabilidades):
    probabilidades = np.asarray(probabilidades, dtype=float)
    probabilidades = probabilidades[probabilidades > 0]

    entropia = -np.sum(probabilidades * np.log2(probabilidades))

    if len(probabilidades) <= 1:
        entropia_normalizada = 0.0
    else:
        entropia_normalizada = entropia / np.log2(len(probabilidades))

    return entropia, entropia_normalizada


def calcular_metricas_baseline(y_true, y_pred):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    balanced_acc = balanced_accuracy_score(
        y_true,
        y_pred
    )

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "mcc": mcc,
        "balanced_accuracy": balanced_acc,
        "npv": npv,
        "fpr": fpr,
        "fnr": fnr
    }


def gerar_html_tabela_metricas(metricas_nao_fraude, metricas_fraude):
    linhas = [
        ("Acurácia", "accuracy"),
        ("Precisão da fraude", "precision"),
        ("Recall da fraude", "recall"),
        ("Especificidade", "specificity"),
        ("F1-score da fraude", "f1"),
        ("MCC", "mcc"),
        ("Balanced Accuracy", "balanced_accuracy"),
        ("NPV", "npv"),
        ("FPR", "fpr"),
        ("FNR", "fnr")
    ]

    html_linhas = ""

    for nome, chave in linhas:
        html_linhas += f"""
        <tr>
            <td>{nome}</td>
            <td>{metricas_nao_fraude[chave]:.6f}</td>
            <td>{metricas_fraude[chave]:.6f}</td>
        </tr>
        """

    html = f"""
    <section class="card">
        <h2>Métricas dos Baselines Triviais</h2>

        <div class="table-wrapper">
            <table>
                <thead>
                    <tr>
                        <th>Métrica</th>
                        <th>Baseline: tudo como Não Fraude</th>
                        <th>Baseline: tudo como Fraude</th>
                    </tr>
                </thead>
                <tbody>
                    {html_linhas}
                </tbody>
            </table>
        </div>

        <p class="interpretacao">
            O baseline que classifica tudo como não fraude costuma apresentar acurácia muito alta em bases extremamente desbalanceadas,
            mas possui recall da fraude igual a zero. Já o baseline que classifica tudo como fraude possui recall máximo para fraude,
            porém erra praticamente todas as observações da classe não fraude. Essa comparação mostra por que a acurácia isolada é inadequada.
        </p>
    </section>
    """

    return html


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def gerar_relatorio_target(
    arquivo_dados="creditcard.csv",
    nome_target=None,
    pasta_saida=".",
    nome_arquivo_saida="relatorio_target.html"
):

    # ========================================================
    # LEITURA DOS DADOS
    # ========================================================

    df = pd.read_csv(arquivo_dados)

    if nome_target is None:
        if "status_fraude" in df.columns:
            nome_target = "status_fraude"

        elif "Class" in df.columns:
            nome_target = "Class"

        else:
            raise ValueError(
                "Não encontrei automaticamente o target. "
                "Informe nome_target='sua_coluna'."
            )

    if nome_target not in df.columns:
        raise ValueError(f"A coluna target '{nome_target}' não existe no arquivo.")

    y = df[nome_target].dropna().astype(int).copy()

    valores_unicos = sorted(y.unique())

    if set(valores_unicos) != {0, 1}:
        raise ValueError(
            f"O target precisa ser binário com valores 0 e 1. Valores encontrados: {valores_unicos}"
        )

    # ========================================================
    # ESTATÍSTICAS BÁSICAS DO TARGET
    # ========================================================

    total = len(y)

    n_nao_fraude = int((y == 0).sum())
    n_fraude = int((y == 1).sum())

    p_nao_fraude = n_nao_fraude / total
    p_fraude = n_fraude / total

    percentual_nao_fraude = p_nao_fraude * 100
    percentual_fraude = p_fraude * 100

    razao_desbalanceamento = n_nao_fraude / n_fraude if n_fraude > 0 else np.inf

    uma_fraude_a_cada = total / n_fraude if n_fraude > 0 else np.inf

    odds_fraude = p_fraude / p_nao_fraude if p_nao_fraude > 0 else np.inf
    odds_contra_fraude = p_nao_fraude / p_fraude if p_fraude > 0 else np.inf

    entropia, entropia_normalizada = calcular_entropia_shannon(
        [p_nao_fraude, p_fraude]
    )

    gini = 1 - (p_nao_fraude ** 2 + p_fraude ** 2)

    erro_classe_majoritaria = min(p_nao_fraude, p_fraude)

    # ========================================================
    # BASELINES
    # ========================================================

    y_baseline_tudo_nao_fraude = np.zeros_like(y)
    y_baseline_tudo_fraude = np.ones_like(y)

    metricas_tudo_nao_fraude = calcular_metricas_baseline(
        y_true=y,
        y_pred=y_baseline_tudo_nao_fraude
    )

    metricas_tudo_fraude = calcular_metricas_baseline(
        y_true=y,
        y_pred=y_baseline_tudo_fraude
    )

    # ========================================================
    # SÉRIE ACUMULADA DE FRAUDES
    # ========================================================

    serie_transacao = np.arange(1, total + 1)
    serie_fraude_acumulada = y.cumsum().to_numpy()

    # ========================================================
    # GRÁFICO 1: FREQUÊNCIA ABSOLUTA
    # ========================================================

    fig_abs = go.Figure()

    fig_abs.add_trace(
        go.Bar(
            x=["Não Fraude", "Fraude"],
            y=[n_nao_fraude, n_fraude],
            text=[
                formatar_inteiro(n_nao_fraude),
                formatar_inteiro(n_fraude)
            ],
            textposition="outside",
            hovertemplate=(
                "Classe: %{x}<br>"
                "Quantidade: %{y:,}"
                "<extra></extra>"
            )
        )
    )

    fig_abs.update_layout(
        height=520,
        margin=dict(l=50, r=30, t=60, b=60),
        title=dict(
            text="Frequência Absoluta das Classes do Target",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Classe",
        yaxis_title="Quantidade",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_abs.update_yaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.30)"
    )

    grafico_abs_html = fig_plotly_to_html(fig_abs)

    # ========================================================
    # GRÁFICO 2: FREQUÊNCIA RELATIVA
    # ========================================================

    fig_rel = go.Figure()

    fig_rel.add_trace(
        go.Bar(
            x=["Não Fraude", "Fraude"],
            y=[percentual_nao_fraude, percentual_fraude],
            text=[
                f"{percentual_nao_fraude:.4f}%",
                f"{percentual_fraude:.4f}%"
            ],
            textposition="outside",
            hovertemplate=(
                "Classe: %{x}<br>"
                "Percentual: %{y:.6f}%"
                "<extra></extra>"
            )
        )
    )

    fig_rel.update_layout(
        height=520,
        margin=dict(l=50, r=30, t=60, b=60),
        title=dict(
            text="Frequência Relativa das Classes do Target",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Classe",
        yaxis_title="Percentual (%)",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_rel.update_yaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.30)"
    )

    grafico_rel_html = fig_plotly_to_html(fig_rel)

    # ========================================================
    # GRÁFICO 3: FREQUÊNCIA ABSOLUTA EM ESCALA LOG
    # ========================================================

    fig_log = go.Figure()

    fig_log.add_trace(
        go.Bar(
            x=["Não Fraude", "Fraude"],
            y=[n_nao_fraude, n_fraude],
            text=[
                formatar_inteiro(n_nao_fraude),
                formatar_inteiro(n_fraude)
            ],
            textposition="outside",
            hovertemplate=(
                "Classe: %{x}<br>"
                "Quantidade: %{y:,}"
                "<extra></extra>"
            )
        )
    )

    fig_log.update_layout(
        height=520,
        margin=dict(l=50, r=30, t=60, b=60),
        title=dict(
            text="Frequência Absoluta em Escala Logarítmica",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Classe",
        yaxis_title="Quantidade em escala log",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_log.update_yaxes(
        type="log",
        showgrid=True,
        gridcolor="rgba(148,163,184,0.30)"
    )

    grafico_log_html = fig_plotly_to_html(fig_log)

    # ========================================================
    # GRÁFICO 4: MATRIZES DE CONFUSÃO DOS BASELINES
    # ========================================================

    cm_nao_fraude = np.array(
        [
            [metricas_tudo_nao_fraude["tn"], metricas_tudo_nao_fraude["fp"]],
            [metricas_tudo_nao_fraude["fn"], metricas_tudo_nao_fraude["tp"]]
        ]
    )

    cm_fraude = np.array(
        [
            [metricas_tudo_fraude["tn"], metricas_tudo_fraude["fp"]],
            [metricas_tudo_fraude["fn"], metricas_tudo_fraude["tp"]]
        ]
    )

    fig_cm = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            "Baseline: Tudo como Não Fraude",
            "Baseline: Tudo como Fraude"
        ],
        horizontal_spacing=0.18
    )

    fig_cm.add_trace(
        go.Heatmap(
            z=cm_nao_fraude,
            x=["Pred Não Fraude", "Pred Fraude"],
            y=["Real Não Fraude", "Real Fraude"],
            text=[
                [formatar_inteiro(cm_nao_fraude[0, 0]), formatar_inteiro(cm_nao_fraude[0, 1])],
                [formatar_inteiro(cm_nao_fraude[1, 0]), formatar_inteiro(cm_nao_fraude[1, 1])]
            ],
            texttemplate="%{text}",
            colorscale="Blues",
            showscale=False,
            hovertemplate=(
                "%{y}<br>"
                "%{x}<br>"
                "Quantidade: %{z:,}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=1
    )

    fig_cm.add_trace(
        go.Heatmap(
            z=cm_fraude,
            x=["Pred Não Fraude", "Pred Fraude"],
            y=["Real Não Fraude", "Real Fraude"],
            text=[
                [formatar_inteiro(cm_fraude[0, 0]), formatar_inteiro(cm_fraude[0, 1])],
                [formatar_inteiro(cm_fraude[1, 0]), formatar_inteiro(cm_fraude[1, 1])]
            ],
            texttemplate="%{text}",
            colorscale="Reds",
            showscale=False,
            hovertemplate=(
                "%{y}<br>"
                "%{x}<br>"
                "Quantidade: %{z:,}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=2
    )

    fig_cm.update_layout(
        height=560,
        margin=dict(l=80, r=40, t=90, b=80),
        title=dict(
            text="Matrizes de Confusão dos Baselines Triviais",
            x=0.5,
            xanchor="center"
        ),
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    grafico_cm_html = fig_plotly_to_html(fig_cm)

    # ========================================================
    # GRÁFICO 5: SÉRIE ACUMULADA DE FRAUDES
    # ========================================================

    fig_serie = go.Figure()

    fig_serie.add_trace(
        go.Scattergl(
            x=serie_transacao,
            y=serie_fraude_acumulada,
            mode="lines",
            name="Fraudes acumuladas",
            line=dict(
                width=2
            ),
            hovertemplate=(
                "Transação: %{x}<br>"
                "Fraudes acumuladas: %{y}"
                "<extra></extra>"
            )
        )
    )

    fig_serie.update_layout(
        height=560,
        margin=dict(l=60, r=30, t=70, b=60),
        title=dict(
            text="Série Acumulada de Fraudes por Ordem da Transação",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Índice da transação",
        yaxis_title="Quantidade acumulada de fraudes",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_serie.update_xaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.25)"
    )

    fig_serie.update_yaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.25)"
    )

    grafico_serie_html = fig_plotly_to_html(fig_serie)

    # ========================================================
    # HTML TABELA DE MÉTRICAS
    # ========================================================

    tabela_metricas_html = gerar_html_tabela_metricas(
        metricas_nao_fraude=metricas_tudo_nao_fraude,
        metricas_fraude=metricas_tudo_fraude
    )

    # ========================================================
    # HTML FINAL
    # ========================================================

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">

        <title>Relatório do Target - {nome_target}</title>

        <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

        <style>
            body {{
                margin: 0;
                padding: 32px;
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
            }}

            .container {{
                max-width: 1450px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                color: #020617;
                margin-bottom: 30px;
            }}

            h2 {{
                text-align: center;
                color: #020617;
                margin-top: 0;
                margin-bottom: 22px;
                font-size: 22px;
            }}

            .card {{
                background: #ffffff;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .grid {{
                display: grid;
                grid-template-columns: repeat(4, 1fr);
                gap: 14px;
            }}

            .metric-card {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 14px;
                padding: 16px;
            }}

            .metric-label {{
                font-size: 13px;
                font-weight: 700;
                color: #475569;
                margin-bottom: 8px;
            }}

            .metric-value {{
                font-size: 22px;
                font-weight: 900;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
                word-break: break-word;
            }}

            .metric-sub {{
                margin-top: 6px;
                font-size: 12px;
                color: #64748b;
                line-height: 1.35;
            }}

            .plotly-graph-div {{
                width: 100% !important;
            }}

            .table-wrapper {{
                overflow-x: auto;
            }}

            table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 15px;
            }}

            th {{
                background: #0f172a;
                color: white;
                padding: 12px;
                text-align: left;
            }}

            td {{
                border-bottom: 1px solid #e2e8f0;
                padding: 12px;
                font-family: Consolas, Monaco, monospace;
            }}

            tr:nth-child(even) {{
                background: #f8fafc;
            }}

            .interpretacao {{
                margin-top: 18px;
                background: #f8fafc;
                border-left: 5px solid #2563eb;
                padding: 14px 16px;
                border-radius: 10px;
                line-height: 1.55;
                color: #334155;
            }}

            .alerta {{
                background: #fff7ed;
                border-left: 5px solid #f97316;
            }}

            @media (max-width: 1100px) {{
                .grid {{
                    grid-template-columns: repeat(2, 1fr);
                }}
            }}

            @media (max-width: 700px) {{
                .grid {{
                    grid-template-columns: 1fr;
                }}

                body {{
                    padding: 16px;
                }}
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1>Relatório Exploratório do Target - {nome_target}</h1>

            <section class="card">
                <h2>Resumo Geral do Target</h2>

                <div class="grid">

                    <div class="metric-card">
                        <div class="metric-label">Total de observações</div>
                        <div class="metric-value">{formatar_inteiro(total)}</div>
                        <div class="metric-sub">Quantidade total de registros válidos no target.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Não fraudes</div>
                        <div class="metric-value">{formatar_inteiro(n_nao_fraude)}</div>
                        <div class="metric-sub">{percentual_nao_fraude:.6f}% da base.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Fraudes</div>
                        <div class="metric-value">{formatar_inteiro(n_fraude)}</div>
                        <div class="metric-sub">{percentual_fraude:.6f}% da base.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Classe majoritária</div>
                        <div class="metric-value">Não Fraude</div>
                        <div class="metric-sub">Classe com maior frequência observada.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Razão de desbalanceamento</div>
                        <div class="metric-value">{formatar_numero(razao_desbalanceamento, 2)} : 1</div>
                        <div class="metric-sub">Aproximadamente {formatar_numero(razao_desbalanceamento, 2)} não fraudes para cada fraude.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Probabilidade empírica de fraude</div>
                        <div class="metric-value">{p_fraude:.8f}</div>
                        <div class="metric-sub">Equivalente a {percentual_fraude:.6f}%.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Frequência interpretável</div>
                        <div class="metric-value">1 a cada {formatar_numero(uma_fraude_a_cada, 2)}</div>
                        <div class="metric-sub">Em média, uma fraude a cada X transações.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Odds contra fraude</div>
                        <div class="metric-value">{formatar_numero(odds_contra_fraude, 2)}</div>
                        <div class="metric-sub">Razão probabilística P(não fraude) / P(fraude).</div>
                    </div>

                </div>

                <p class="interpretacao alerta">
                    A variável target apresenta forte desbalanceamento. Isso significa que métricas globais, como acurácia,
                    podem ser enganosas quando analisadas isoladamente. Um classificador trivial que sempre prediz a classe majoritária
                    pode alcançar alta acurácia sem detectar nenhuma fraude.
                </p>
            </section>

            <section class="card">
                {grafico_abs_html}
            </section>

            <section class="card">
                {grafico_rel_html}
            </section>

            <section class="card">
                {grafico_log_html}
            </section>

            <section class="card">
                <h2>Medidas de Incerteza e Concentração do Target</h2>

                <div class="grid">

                    <div class="metric-card">
                        <div class="metric-label">Entropia de Shannon</div>
                        <div class="metric-value">{entropia:.8f}</div>
                        <div class="metric-sub">Medida de incerteza da distribuição binária do target.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Entropia normalizada</div>
                        <div class="metric-value">{entropia_normalizada:.8f}</div>
                        <div class="metric-sub">Entropia dividida pela entropia máxima possível para duas classes.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Impureza de Gini</div>
                        <div class="metric-value">{gini:.8f}</div>
                        <div class="metric-sub">Baixo Gini indica forte concentração em uma classe.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Erro da classe majoritária</div>
                        <div class="metric-value">{erro_classe_majoritaria:.8f}</div>
                        <div class="metric-sub">Taxa de erro se sempre prever a classe majoritária.</div>
                    </div>

                </div>

                <p class="interpretacao">
                    A entropia de Shannon mede a incerteza associada ao target. Como a classe fraude é rara,
                    a distribuição apresenta baixa incerteza global: ao observar apenas a distribuição marginal,
                    a classe não fraude domina quase todos os registros.
                </p>
            </section>

            <section class="card">
                {grafico_cm_html}
            </section>

            {tabela_metricas_html}

            <section class="card">
                {grafico_serie_html}

                <p class="interpretacao">
                    A série acumulada soma +1 sempre que uma transação é fraude e soma 0 quando é não fraude.
                    Assim, o eixo Y representa o total acumulado de fraudes até cada posição da base.
                    Trechos mais inclinados indicam regiões da ordenação em que fraudes aparecem com maior frequência.
                </p>
            </section>

        </div>
    </body>
    </html>
    """

    # ========================================================
    # SALVAR HTML
    # ========================================================

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_saida = pasta_saida / nome_arquivo_saida

    caminho_saida.write_text(
        html_final,
        encoding="utf-8"
    )

    print(f"Relatório HTML gerado com sucesso: {caminho_saida.resolve()}")

    return caminho_saida

In [5]:
# ============================================================
# IMPORTAÇÕES
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import confusion_matrix

import plotly.graph_objects as go
import plotly.io as pio


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def formatar_numero(valor, casas=2):
    if pd.isna(valor):
        return "NA"

    if np.isinf(valor):
        return "Infinito"

    return f"{valor:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def formatar_inteiro(valor):
    return f"{int(valor):,}".replace(",", ".")


def fig_plotly_to_html(fig):
    return pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs=False,
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "responsive": True,
            "displaylogo": False,
            "modeBarButtonsToRemove": ["lasso2d", "select2d"]
        }
    )


def calcular_entropia_shannon(probabilidades):
    probabilidades = np.asarray(probabilidades, dtype=float)
    probabilidades = probabilidades[probabilidades > 0]

    entropia = -np.sum(probabilidades * np.log2(probabilidades))

    if len(probabilidades) <= 1:
        entropia_normalizada = 0.0
    else:
        entropia_normalizada = entropia / np.log2(len(probabilidades))

    return entropia, entropia_normalizada


def calcular_matriz_confusao(y_true, y_pred):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


def preparar_valores_matriz(cm):
    tn = cm["tn"]
    fp = cm["fp"]
    fn = cm["fn"]
    tp = cm["tp"]

    total_fraudes = fn + tp
    total_nao_fraudes = tn + fp

    fn_pct = fn / total_fraudes * 100 if total_fraudes > 0 else 0
    tp_pct = tp / total_fraudes * 100 if total_fraudes > 0 else 0

    tn_pct = tn / total_nao_fraudes * 100 if total_nao_fraudes > 0 else 0
    fp_pct = fp / total_nao_fraudes * 100 if total_nao_fraudes > 0 else 0

    return {
        "fn": {
            "pct": fn_pct,
            "count": fn,
            "qualidade": 100 - fn_pct
        },
        "tp": {
            "pct": tp_pct,
            "count": tp,
            "qualidade": tp_pct
        },
        "tn": {
            "pct": tn_pct,
            "count": tn,
            "qualidade": tn_pct
        },
        "fp": {
            "pct": fp_pct,
            "count": fp,
            "qualidade": 100 - fp_pct
        }
    }


def preparar_valores_matriz_ideal(n_nao_fraude, n_fraude):
    return {
        "fn": {
            "pct": 0.0,
            "count": 0,
            "qualidade": 100.0
        },
        "tp": {
            "pct": 100.0,
            "count": n_fraude,
            "qualidade": 100.0
        },
        "tn": {
            "pct": 100.0,
            "count": n_nao_fraude,
            "qualidade": 100.0
        },
        "fp": {
            "pct": 0.0,
            "count": 0,
            "qualidade": 100.0
        }
    }


def cor_por_qualidade(q):
    if q >= 95:
        return "cell q95"
    elif q >= 85:
        return "cell q85"
    elif q >= 70:
        return "cell q70"
    elif q >= 50:
        return "cell q50"
    elif q >= 30:
        return "cell q30"
    else:
        return "cell q10"


def gerar_html_matriz(titulo, valores, matriz_ideal=False):
    if matriz_ideal:
        desc_fn = "Ideal: nenhuma fraude perdida"
        desc_tp = "Ideal: fraudes detectadas"
        desc_tn = "Ideal: não fraudes corretas"
        desc_fp = "Ideal: nenhum falso alerta"
    else:
        desc_fn = "Erro: fraude perdida"
        desc_tp = "Acerto: fraude detectada"
        desc_tn = "Acerto: não fraude"
        desc_fp = "Erro: falso alerta"

    html = f"""
    <section class="matrix-card">
        <h2>{titulo}</h2>

        <div class="matrix-area">

            <div class="matrix-wrapper">

                <div class="corner"></div>
                <div class="x-label">Pred Não Fraude</div>
                <div class="x-label">Pred Fraude</div>

                <div class="y-label">Real Fraude</div>

                <div class="{cor_por_qualidade(valores['fn']['qualidade'])}">
                    <div class="pct">{valores['fn']['pct']:.2f}%</div>
                    <div class="count">({formatar_inteiro(valores['fn']['count'])})</div>
                    <div class="cell-desc">{desc_fn}</div>
                </div>

                <div class="{cor_por_qualidade(valores['tp']['qualidade'])}">
                    <div class="pct">{valores['tp']['pct']:.2f}%</div>
                    <div class="count">({formatar_inteiro(valores['tp']['count'])})</div>
                    <div class="cell-desc">{desc_tp}</div>
                </div>

                <div class="y-label">Real Não Fraude</div>

                <div class="{cor_por_qualidade(valores['tn']['qualidade'])}">
                    <div class="pct">{valores['tn']['pct']:.2f}%</div>
                    <div class="count">({formatar_inteiro(valores['tn']['count'])})</div>
                    <div class="cell-desc">{desc_tn}</div>
                </div>

                <div class="{cor_por_qualidade(valores['fp']['qualidade'])}">
                    <div class="pct">{valores['fp']['pct']:.2f}%</div>
                    <div class="count">({formatar_inteiro(valores['fp']['count'])})</div>
                    <div class="cell-desc">{desc_fp}</div>
                </div>

            </div>

            <div class="quality-legend">
                <div class="quality-title">Qualidade</div>
                <div class="quality-bar"></div>
                <div class="quality-top">Melhor</div>
                <div class="quality-bottom">Pior</div>
            </div>

        </div>
    </section>
    """

    return html


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def gerar_relatorio_target(
    arquivo_dados="creditcard.csv",
    nome_target=None,
    pasta_saida=".",
    nome_arquivo_saida="relatorio_target.html"
):

    # ========================================================
    # LEITURA DOS DADOS
    # ========================================================

    df = pd.read_csv(arquivo_dados)

    if nome_target is None:
        if "status_fraude" in df.columns:
            nome_target = "status_fraude"

        elif "Class" in df.columns:
            nome_target = "Class"

        else:
            raise ValueError(
                "Não encontrei automaticamente o target. "
                "Informe nome_target='sua_coluna'."
            )

    if nome_target not in df.columns:
        raise ValueError(f"A coluna target '{nome_target}' não existe no arquivo.")

    y = df[nome_target].dropna().astype(int).copy()

    valores_unicos = sorted(y.unique())

    if set(valores_unicos) != {0, 1}:
        raise ValueError(
            f"O target precisa ser binário com valores 0 e 1. "
            f"Valores encontrados: {valores_unicos}"
        )

    # ========================================================
    # ESTATÍSTICAS BÁSICAS DO TARGET
    # ========================================================

    total = len(y)

    n_nao_fraude = int((y == 0).sum())
    n_fraude = int((y == 1).sum())

    p_nao_fraude = n_nao_fraude / total
    p_fraude = n_fraude / total

    percentual_nao_fraude = p_nao_fraude * 100
    percentual_fraude = p_fraude * 100

    if n_nao_fraude >= n_fraude:
        classe_majoritaria = "Não Fraude"
        classe_minoritaria = "Fraude"
        n_majoritaria = n_nao_fraude
        n_minoritaria = n_fraude
    else:
        classe_majoritaria = "Fraude"
        classe_minoritaria = "Não Fraude"
        n_majoritaria = n_fraude
        n_minoritaria = n_nao_fraude

    razao_desbalanceamento = (
        n_majoritaria / n_minoritaria
        if n_minoritaria > 0
        else np.inf
    )

    entropia, entropia_normalizada = calcular_entropia_shannon(
        [p_nao_fraude, p_fraude]
    )

    # ========================================================
    # BASELINES TRIVIAIS
    # ========================================================

    y_baseline_tudo_nao_fraude = np.zeros_like(y)
    y_baseline_tudo_fraude = np.ones_like(y)

    cm_tudo_nao_fraude = calcular_matriz_confusao(
        y_true=y,
        y_pred=y_baseline_tudo_nao_fraude
    )

    cm_tudo_fraude = calcular_matriz_confusao(
        y_true=y,
        y_pred=y_baseline_tudo_fraude
    )

    valores_tudo_nao_fraude = preparar_valores_matriz(
        cm_tudo_nao_fraude
    )

    valores_tudo_fraude = preparar_valores_matriz(
        cm_tudo_fraude
    )

    valores_ideal = preparar_valores_matriz_ideal(
        n_nao_fraude=n_nao_fraude,
        n_fraude=n_fraude
    )

    html_matriz_tudo_nao_fraude = gerar_html_matriz(
        titulo="Matriz de Confusão (%) - Baseline Trivial: Tudo como Não Fraude",
        valores=valores_tudo_nao_fraude
    )

    html_matriz_tudo_fraude = gerar_html_matriz(
        titulo="Matriz de Confusão (%) - Baseline Trivial: Tudo como Fraude",
        valores=valores_tudo_fraude
    )

    html_matriz_ideal = gerar_html_matriz(
        titulo="Matriz de Confusão (%) - Cenário Ideal",
        valores=valores_ideal,
        matriz_ideal=True
    )

    # ========================================================
    # SÉRIE ACUMULADA DE FRAUDES
    # ========================================================

    serie_transacao = np.arange(1, total + 1)
    serie_fraude_acumulada = y.cumsum().to_numpy()

    # ========================================================
    # GRÁFICO 1: FREQUÊNCIA ABSOLUTA
    # ========================================================

    fig_abs = go.Figure()

    fig_abs.add_trace(
        go.Bar(
            x=["Não Fraude", "Fraude"],
            y=[n_nao_fraude, n_fraude],
            text=[
                formatar_inteiro(n_nao_fraude),
                formatar_inteiro(n_fraude)
            ],
            textposition="outside",
            marker=dict(
                color=["#2563eb", "#facc15"],
                line=dict(
                    color="#111827",
                    width=1
                )
            ),
            hovertemplate=(
                "Classe: %{x}<br>"
                "Quantidade: %{y:,}"
                "<extra></extra>"
            )
        )
    )

    fig_abs.update_layout(
        height=520,
        margin=dict(l=50, r=30, t=60, b=60),
        title=dict(
            text="Frequência Absoluta das Classes do Target",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Classe",
        yaxis_title="Quantidade",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_abs.update_yaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.30)"
    )

    grafico_abs_html = fig_plotly_to_html(fig_abs)

    # ========================================================
    # GRÁFICO 2: FREQUÊNCIA RELATIVA HORIZONTAL
    # ========================================================

    fig_rel = go.Figure()

    fig_rel.add_trace(
        go.Bar(
            y=["Não Fraude", "Fraude"],
            x=[percentual_nao_fraude, percentual_fraude],
            orientation="h",
            text=[
                f"{percentual_nao_fraude:.4f}%",
                f"{percentual_fraude:.4f}%"
            ],
            textposition="outside",
            marker=dict(
                color=["#2563eb", "#facc15"],
                line=dict(
                    color="#111827",
                    width=1
                )
            ),
            hovertemplate=(
                "Classe: %{y}<br>"
                "Percentual: %{x:.6f}%"
                "<extra></extra>"
            )
        )
    )

    fig_rel.update_layout(
        height=520,
        margin=dict(l=110, r=80, t=60, b=60),
        title=dict(
            text="Frequência Relativa das Classes do Target",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Percentual (%)",
        yaxis_title="Classe",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_rel.update_xaxes(
        range=[0, max(105, percentual_nao_fraude * 1.12)],
        showgrid=True,
        gridcolor="rgba(148,163,184,0.30)"
    )

    fig_rel.update_yaxes(
        autorange="reversed"
    )

    grafico_rel_html = fig_plotly_to_html(fig_rel)

    # ========================================================
    # GRÁFICO 3: SÉRIE ACUMULADA DE FRAUDES
    # ========================================================

    fig_serie = go.Figure()

    fig_serie.add_trace(
        go.Scattergl(
            x=serie_transacao,
            y=serie_fraude_acumulada,
            mode="lines",
            name="Fraudes acumuladas",
            line=dict(
                width=2,
                color="#dc2626"
            ),
            hovertemplate=(
                "Transação: %{x}<br>"
                "Fraudes acumuladas: %{y}"
                "<extra></extra>"
            )
        )
    )

    fig_serie.update_layout(
        height=560,
        margin=dict(l=60, r=30, t=70, b=60),
        title=dict(
            text="Série Acumulada de Fraudes por Ordem da Transação",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Índice da transação",
        yaxis_title="Quantidade acumulada de fraudes",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_serie.update_xaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.25)"
    )

    fig_serie.update_yaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.25)"
    )

    grafico_serie_html = fig_plotly_to_html(fig_serie)

    # ========================================================
    # HTML FINAL
    # ========================================================

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">

        <title>Relatório do Target - {nome_target}</title>

        <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

        <style>
            body {{
                margin: 0;
                padding: 32px;
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
            }}

            .container {{
                max-width: 1450px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                color: #020617;
                margin-bottom: 30px;
            }}

            h2 {{
                text-align: center;
                color: #020617;
                margin-top: 0;
                margin-bottom: 22px;
                font-size: 22px;
            }}

            .card,
            .matrix-card {{
                background: #ffffff;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .grid {{
                display: grid;
                grid-template-columns: repeat(4, 1fr);
                gap: 14px;
            }}

            .metric-card {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 14px;
                padding: 16px;
            }}

            .metric-label {{
                font-size: 13px;
                font-weight: 700;
                color: #475569;
                margin-bottom: 8px;
            }}

            .metric-value {{
                font-size: 22px;
                font-weight: 900;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
                word-break: break-word;
            }}

            .metric-sub {{
                margin-top: 6px;
                font-size: 12px;
                color: #64748b;
                line-height: 1.35;
            }}

            .plotly-graph-div {{
                width: 100% !important;
            }}

            .interpretacao {{
                margin-top: 18px;
                background: #f8fafc;
                border-left: 5px solid #2563eb;
                padding: 14px 16px;
                border-radius: 10px;
                line-height: 1.55;
                color: #334155;
            }}

            .alerta {{
                background: #fff7ed;
                border-left: 5px solid #f97316;
            }}

            .matrix-area {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 34px;
            }}

            .matrix-wrapper {{
                display: grid;
                grid-template-columns: 180px 1fr 1fr;
                grid-template-rows: 48px 190px 190px;
                width: 950px;
            }}

            .corner {{
                background: transparent;
            }}

            .x-label {{
                display: flex;
                align-items: center;
                justify-content: center;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
                border-bottom: 1px solid #e5e7eb;
            }}

            .y-label {{
                display: flex;
                align-items: center;
                justify-content: flex-end;
                padding-right: 18px;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
            }}

            .cell {{
                display: flex;
                flex-direction: column;
                align-items: center;
                justify-content: center;
                min-height: 180px;
                border: 1px solid #e5e7eb;
                font-size: 20px;
                text-align: center;
                color: #020617 !important;
            }}

            .pct {{
                font-size: 30px;
                font-weight: 900;
                margin-bottom: 4px;
                color: #020617 !important;
            }}

            .count {{
                font-size: 24px;
                font-weight: 900;
                margin-bottom: 8px;
                color: #020617 !important;
            }}

            .cell-desc {{
                font-size: 13px;
                font-weight: 700;
                opacity: 1;
                color: #020617 !important;
            }}

            .q95 {{
                background: #08306b;
            }}

            .q85 {{
                background: #08519c;
            }}

            .q70 {{
                background: #2171b5;
            }}

            .q50 {{
                background: #6baed6;
            }}

            .q30 {{
                background: #c6dbef;
            }}

            .q10 {{
                background: #eff6ff;
            }}

            .quality-legend {{
                position: relative;
                display: flex;
                flex-direction: column;
                align-items: center;
                min-width: 115px;
            }}

            .quality-title {{
                font-weight: 800;
                font-size: 15px;
                margin-bottom: 10px;
                color: #020617;
            }}

            .quality-bar {{
                width: 30px;
                height: 310px;
                border-radius: 16px;
                background: linear-gradient(
                    to bottom,
                    #08306b 0%,
                    #08519c 18%,
                    #2171b5 36%,
                    #6baed6 58%,
                    #c6dbef 78%,
                    #eff6ff 100%
                );
                border: 1px solid #cbd5e1;
            }}

            .quality-top {{
                position: absolute;
                top: 43px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .quality-bottom {{
                position: absolute;
                top: 335px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            @media (max-width: 1100px) {{
                .grid {{
                    grid-template-columns: repeat(2, 1fr);
                }}

                .matrix-area {{
                    flex-direction: column;
                }}

                .matrix-wrapper {{
                    width: 100%;
                    grid-template-columns: 150px 1fr 1fr;
                }}
            }}

            @media (max-width: 700px) {{
                .grid {{
                    grid-template-columns: 1fr;
                }}

                body {{
                    padding: 16px;
                }}

                .matrix-wrapper {{
                    grid-template-columns: 120px 1fr 1fr;
                    grid-template-rows: 48px 160px 160px;
                }}

                .pct {{
                    font-size: 22px;
                }}

                .count {{
                    font-size: 18px;
                }}

                .y-label,
                .x-label {{
                    font-size: 14px;
                }}
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1>Relatório Exploratório do Target - {nome_target}</h1>

            <section class="card">
                <h2>Resumo Geral do Target</h2>

                <div class="grid">

                    <div class="metric-card">
                        <div class="metric-label">Total de observações</div>
                        <div class="metric-value">{formatar_inteiro(total)}</div>
                        <div class="metric-sub">Quantidade total de registros válidos no target.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Não fraudes</div>
                        <div class="metric-value">{formatar_inteiro(n_nao_fraude)}</div>
                        <div class="metric-sub">{percentual_nao_fraude:.6f}% da base.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Fraudes</div>
                        <div class="metric-value">{formatar_inteiro(n_fraude)}</div>
                        <div class="metric-sub">{percentual_fraude:.6f}% da base.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Classe majoritária</div>
                        <div class="metric-value">{classe_majoritaria}</div>
                        <div class="metric-sub">{formatar_inteiro(n_majoritaria)} observações.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Classe minoritária</div>
                        <div class="metric-value">{classe_minoritaria}</div>
                        <div class="metric-sub">{formatar_inteiro(n_minoritaria)} observações.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Razão de desbalanceamento</div>
                        <div class="metric-value">{formatar_numero(razao_desbalanceamento, 2)} : 1</div>
                        <div class="metric-sub">Razão entre classe majoritária e classe minoritária.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Probabilidade empírica de fraude</div>
                        <div class="metric-value">{p_fraude:.8f}</div>
                        <div class="metric-sub">Equivalente a {percentual_fraude:.6f}%.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Entropia de Shannon</div>
                        <div class="metric-value">{entropia:.8f}</div>
                        <div class="metric-sub">Incerteza da distribuição binária do target.</div>
                    </div>

                </div>

                <p class="interpretacao alerta">
                    A variável target apresenta forte desbalanceamento entre as classes.
                    A classe minoritária é muito menos frequente, o que torna inadequado avaliar modelos apenas por acurácia.
                    A entropia de Shannon também reforça a concentração da distribuição em uma única classe.
                </p>
            </section>

            <section class="card">
                {grafico_abs_html}
            </section>

            <section class="card">
                {grafico_rel_html}
            </section>

            {html_matriz_ideal}

            {html_matriz_tudo_nao_fraude}

            {html_matriz_tudo_fraude}

            <section class="card">
                {grafico_serie_html}

                <p class="interpretacao">
                    A série acumulada soma +1 sempre que uma transação é fraude e soma 0 quando é não fraude.
                    Assim, o eixo Y representa o total acumulado de fraudes até cada posição da base.
                    Trechos mais inclinados indicam regiões da ordenação em que fraudes aparecem com maior frequência.
                </p>
            </section>

        </div>
    </body>
    </html>
    """

    # ========================================================
    # SALVAR HTML
    # ========================================================

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_saida = pasta_saida / nome_arquivo_saida

    caminho_saida.write_text(
        html_final,
        encoding="utf-8"
    )

    print(f"Relatório HTML gerado com sucesso: {caminho_saida.resolve()}")

    return caminho_saida

In [6]:
gerar_relatorio_target(
    arquivo_dados="creditcard.csv",
    nome_target=None,
    pasta_saida=".",
    nome_arquivo_saida="relatorio_target.html"
)

Relatório HTML gerado com sucesso: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\relatorio_target.html


WindowsPath('relatorio_target.html')